<div dir="rtl">

# تمرین پایانی ۱۳
# از شمارش کلمات تا ترنسفورمر

این نوت‌بوک **مسیر کار** است، نه راه‌حل. صورت کامل سؤال‌ها در فایل PDF است:

`week13_hw_projects.pdf`

در هر بخش:
1. اول markdown همان بخش را بخوانید.
2. سلول کد را خودتان پر کنید.
3. توضیح کوتاه را در سلول «توضیح این بخش» بنویسید.

کد آماده را از کلاس کپی نکنید. فقط ایده بگیرید.


<style>
.jp-MarkdownCell .jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.text_cell_render,
.rendered-markdown,
[dir="rtl"] {
  direction: rtl;
  text-align: right;
}
[dir="rtl"] p,
[dir="rtl"] li,
[dir="rtl"] ul,
[dir="rtl"] ol,
[dir="rtl"] h1,
[dir="rtl"] h2,
[dir="rtl"] h3,
[dir="rtl"] h4,
[dir="rtl"] blockquote,
[dir="rtl"] table {
  text-align: right;
}
[dir="rtl"] code {
  direction: ltr;
  unicode-bidi: isolate;
}
[dir="rtl"] pre,
[dir="rtl"] pre code {
  direction: ltr;
  unicode-bidi: isolate;
  text-align: left;
}
[dir="rtl"] blockquote {
  border-left: none;
  border-right: 4px solid #d0d7de;
  padding-left: 0;
  padding-right: 1em;
}
[dir="ltr"] {
  unicode-bidi: isolate;
}
</style>

</div>


<div dir="rtl">

## آماده‌سازی

</div>


In [20]:
# اگر روی Colab هستید و بسته‌ای کم دارید، این خط را از حالت توضیح خارج کنید.
# %pip install torch scikit-learn transformers -q

import math
import re
from collections import Counter
from sklearn.datasets import fetch_20newsgroups

import torch
from torch import nn

torch.manual_seed(0)


<div dir="rtl">

# پروژه ۱. خبرخوان کوچک: از شمارش کلمات تا Encoder

**هدف:** یک طبقه‌بند کوچک برای دو موضوع خبری بسازید و ببینید BoW، embedding و Encoder هر کدام چه اطلاعاتی می‌بینند.

**داده:** `20 Newsgroups` فقط دو کلاس، حجم عمداً کوچک.

</div>


<div dir="rtl">

## بخش ۱ـ الف. داده، تمیزکاری و نگاه اول

باید انجام دهید:
- دو کلاس `sci.space` و `rec.sport.hockey` را بارگذاری کنید.
- حداکثر ۲۵۰ سند آموزش و ۱۰۰ سند آزمون بردارید.
- تابع نرمال‌سازی بنویسید.
- تعداد نمونه، میانگین طول، و دو نمونهٔ خام در کنار نسخهٔ تمیز را چاپ کنید.

</div>


In [21]:
CATEGORIES = ["sci.space", "rec.sport.hockey"]

MAX_TRAIN = 250
MAX_TEST = 100


def normalize_text(text: str) -> str:
    ### START CODE HERE ###
    # lowercase, remove URLs, drop extra characters, collapse spaces
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text) # remove URLs
    text = re.sub(r"[^a-z\s]"," ",text) # remove everything else a-z
    text = re.sub(r"\s+"," ",text) # remove all extra white spaces.
    return text
    ### END CODE HERE ###

### START CODE HERE ###
# 1) load 20 Newsgroups with sklearn (only CATEGORIES)
train_raw = fetch_20newsgroups(subset="train", categories=CATEGORIES,remove=("headers","footers","quotes"), random_state=42)
#
test_raw = fetch_20newsgroups(subset="test", categories=CATEGORIES,remove=("headers","footers","quotes"), random_state=42)
#
X_train_raw = train_raw.data[:250]
y_train_raw = train_raw.target[:250]

X_test_raw = test_raw.data[:100]
y_test_raw = test_raw.target[:100]


# 2) keep at most MAX_TRAIN / MAX_TEST documents
# 3) optionally remove headers/footers/quotes
# 4) apply normalize_text
X_train_clean = [normalize_text(text) for text in X_train_raw]
X_test_clean = [normalize_text(text) for text in X_test_raw]


train_texts, train_labels = X_train_clean, y_train_raw
test_texts, test_labels = X_test_clean, y_test_raw

# ### END CODE HERE ###
# data = fetch_20newsgroups()
# df = data.data
print("train size:", train_labels.size)
print("test size:", test_labels.size)
print("class counts:\n", Counter(train_labels))
print("mean train length (words):", (sum([len(text.split()) for text in train_texts])/train_labels.size)                     )
print("RAW :", (sum([len(text.split()) for text in X_train_raw])/train_labels.size))
print("CLEAN:", (sum([len(text.split()) for text in X_train_clean])/train_labels.size))


train size: 250
test size: 100
class counts:
 Counter({np.int64(0): 126, np.int64(1): 124})
mean train length (words): 226.168
RAW : 261.32
CLEAN: 226.168


<div dir="rtl">

### توضیح بخش ۱ـ الف

- تعداد نمونه در هر کلاس:فضا 126 نمونه ، هاکی 126
- میانگین طول:226
- مشاهدهٔ شما از خام در برابر تمیز: داده خام فوق العاده طولانی پر از عدد و  ساختار جدولی بود ولی داده تمیز صرفا توالی از متن های متعارف بود ( فاقد لینک و کاراکتر ها و اعداد ) البته هنگام فراخوانی هدر و فوتر و کوتز ها رو حذف کردم



</div>


<div dir="rtl">

## بخش ۱ـ ب. خط پایهٔ شمارش کلمات

باید انجام دهید:
- BoW یا TF-IDF
- یک مدل خطی مثل Logistic Regression
- دقت آموزش و آزمون
- دو سند اشتباه
- دو جملهٔ خودتان که زیر BoW یکی می‌شوند

</div>


In [22]:
### START CODE HERE ###
from sklearn.linear_model import LogisticRegression
import numpy as np

def get_bow_vector(text,vocab_size,stoi):
    vector = np.zeros(vocab_size)
    for word in text.split():
        if word in stoi:
            idx = stoi[word]
            #count each word
            vector[idx] += 1
    return vector

all_words = set()
for text in train_texts:
    for word in text.split():
        all_words.add(word)

vocab = sorted(list(all_words))
stoi = {token:i for i ,token in enumerate(vocab)}


X_train_bow = np.array([get_bow_vector(text,len(vocab),stoi) for text in train_texts])
X_test_bow = np.array([get_bow_vector(text,len(vocab),stoi) for text in test_texts])


model = LogisticRegression(random_state=42)

model.fit(X_train_bow, train_labels)

# vectorize clean train/test texts
# fit a linear classifier
# print train accuracy and test accuracy

# vectorizer = None
# bow_model = None
train_acc = model.score(X_train_bow, train_labels)
test_acc = model.score(X_test_bow, test_labels)
# ### END CODE HERE ###
#
print("BoW train acc:", train_acc)
print("BoW test acc :", test_acc)

BoW train acc: 1.0
BoW test acc : 0.88


In [23]:
### START CODE HERE ###
# find two test documents the BoW model gets wrong
# print a short slice of each and the true/predicted labels
### END CODE HERE ###
import math

my_pair = [
    "WRITE SENTENCE A",
    "WRITE SENTENCE B",
]
same_bow = [
    "Are you BOW?",
    "You are BOW."
]
### START CODE HERE ###
# show that these two sentences get the same BoW vector
### END CODE HERE ###
for test in my_pair:
    test = normalize_text(test)
    bow = get_bow_vector(test,len(vocab),stoi)
    print(f"bow for \n{bow}")

result =[]
for test in same_bow:
    test = normalize_text(test)
    result.append(get_bow_vector(test,len(vocab),stoi))

print(f"result:\n{same_bow}")
assert sum(result[0]) == sum(result[1]), ("NE sum pair 1 and pair 2")
print("sum pair 1 and pair2 is Equal")

bow for 
[1. 0. 0. ... 0. 0. 0.]
bow for 
[0. 0. 0. ... 0. 0. 0.]
result:
['Are you BOW?', 'You are BOW.']
sum pair 1 and pair2 is Equal


<div dir="rtl">

### توضیح بخش ۱ ـ ب

- دو خطای مدل و حدس شما:ناتوانی در تشخیص ترتیب ، طول بسیار زیاد بردار در صورت زیاد بودن اندازه vocab در ضمن توان درک ارتباط مفهومی کلمات یک sequence با هم را ندارد
- این دو جمله چرا زیر BoW یکی می‌شوند؟ چون صرفا ترتیب متفاوتی از چینش دارند و این روش نسبت به ترتیب نابینا هست
- BoW چه چیزی را می‌بیند و چه چیزی را نمی‌بیند؟ فراوانی هر کلمه را میبیند ولی ترتیب را نمی تواند درک کند همچنین ارتباط مفهومی هم نمی تواند درک کند

</div>


<div dir="rtl">

## بخش ۱ـ ج. از شناسهٔ کلمه به embedding

باید انجام دهید:
- واژه‌نامه از دادهٔ آموزش؛ کلمات نادر → `unk`
- هر سند → دنبالهٔ ID با طول ثابت
- `nn.Embedding` + میانگین توکن‌ها + لایهٔ خطی ۲ کلاسه
- چند epoch آموزش و مقایسه با BoW
- حدود ۱۰ جفت skip-gram با پنجرهٔ ۲

</div>


In [24]:
MAX_LEN = 64
MIN_FREQ = 2
UNK = "<unk>"
PAD = "<pad>"

### START CODE HERE ###
# build vocab from train_texts
freq = []
for text in train_texts:
    for word in text.split():
        freq.append(word)
all_words = set()
for text in train_texts:
    for word in text.split():
        all_words.add(word)

all_words = sorted(list(all_words))
vocab = [PAD,UNK]
vocab+= all_words
# encode a list of documents to a LongTensor of shape [N, MAX_LEN]
stoi = {token:i for i,token in enumerate(vocab)}
itos = {i:token for i,token in enumerate(vocab)}


def encode_documents(texts,max_len=MAX_LEN):
    counter = Counter(freq)
    ids = [
        stoi[token] if counter.get(token,1)>=MIN_FREQ else stoi[UNK]
           for token in texts.split()
    ]

    ids = ids[:max_len]
    ids += [stoi[PAD]]*(max_len-len(ids))
    return ids


X_train_ids = [encode_documents(text) for text in train_texts]
X_test_ids = [encode_documents(text) for text in test_texts]
y_train = y_train_raw
y_test = y_test_raw
### END CODE HERE ###

print("vocab size:", len(vocab))
print("X_train_ids.shape:", np.array(X_train_ids).shape)
print("X_train_ids sample:", X_train_ids[1])

vocab size: 8719
X_train_ids.shape: (250, 64)
X_train_ids sample: [8514, 5228, 3618, 723, 900, 7752, 2021, 308, 3903, 3190, 5831, 6749, 2878, 2567, 347, 7682, 3303, 7786, 6291, 7773, 3303, 8501, 3898, 7752, 6706, 460, 7754, 5831, 7908, 7752, 3903, 308, 2021, 6294, 426, 900, 3914, 3898, 7053, 1, 308, 3877, 5192, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [25]:
class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_classes=2, pad_id=0):
        super().__init__()
        ### START CODE HERE ###
        self.pad_id = pad_id
        self.embed = nn.Embedding(vocab_size,d_model,padding_idx=pad_id)

        self.classifier = nn.Linear(d_model,num_classes)
        ### END CODE HERE ###

    def forward(self, token_ids):
        ### START CODE HERE ###
        mask = token_ids.ne(self.pad_id).unsqueeze(-1) # [B , T , apply mask]
        embedding = self.embed(token_ids)

        sum_embed = (embedding*mask).sum(dim=1)
        length = mask.sum(dim=1).clamp(min=1) # avoid zero deviation = each length at least will be equal to 1
        mean_embed = sum_embed/length

        logit = self.classifier(mean_embed)
        return logit
        ### END CODE HERE ###


# make batchs
def make_batch(x,y):
    input_ids = torch.tensor(x,dtype=torch.long)
    labels = torch.tensor(y,dtype=torch.long)
    return input_ids, labels


### START CODE HERE ###
mean_emb_model = MeanEmbeddingClassifier(vocab_size=len(vocab))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mean_emb_model.parameters(), lr=0.005)
epochs = 250

def make_batch (x,y):
    input_ids = torch.tensor(x,dtype= torch.long)
    labels = torch.tensor(y,dtype=torch.long)
    return input_ids,labels


X_train_ids , y_train = make_batch(X_train_ids,y_train)
X_test_ids , y_test = make_batch(X_test_ids,y_test)

emb_test_acc = []

for epoch in range(epochs):
    mean_emb_model.train()
    optimizer.zero_grad()
    logits = mean_emb_model(X_train_ids)  # Shape: [N_train, 2]
    loss = loss_fn(logits, y_train)  # Shape target: [N_train]
    loss.backward()
    optimizer.step()

    mean_emb_model.eval()
    with torch.no_grad():
        test_logits = mean_emb_model(X_test_ids)  # Shape: [N_test, 2]
        preds = test_logits.argmax(dim=-1)
        acc = (preds == y_test).float().mean().item()
        emb_test_acc.append(acc)

### END CODE HERE ###

print("embedding test acc:", emb_test_acc[-1])


embedding test acc: 0.8199999928474426


In [41]:
### START CODE HERE ###
# from the clean training text, build ~10 skip-gram pairs with window=2
# print (center, context) as words, not only IDs
window_size=2
skipgram_pairs = []

for center_index, center_word  in enumerate(vocab):
    left = max(0,center_index-window_size)
    right = min(len(vocab),center_index+window_size+1)

    for context_index in range(left,right):
        if context_index != center_index:
            skipgram_pairs.append([itos[center_index],itos[context_index]])


### END CODE HERE ###
#
print("skip-gram pairs:")
for center, context in skipgram_pairs[-10:]:
    print(center, "->", context)


skip-gram pairs:
zwak -> zware
zwakke -> zwaartepunten
zwakke -> zwak
zwakke -> zware
zwakke -> zwarte
zware -> zwak
zware -> zwakke
zware -> zwarte
zwarte -> zwakke
zwarte -> zware


<div dir="rtl">

### توضیح بخش ۱ـ ج

- embedding نسبت به BoW چه چیزی اضافه کرد؟ بردار تولید شده dense است و مثل bow به طول vocab  نیست ! معنای همسایگی و حضور در بافت های مشابه رو درک میکنه ، اما هنوز بدون اضافه کردن پوزیشنال انکودینگ متوجه ترتیب نمی تونه بشه
- این جفت‌های skip-gram چه سیگنال آموزشی‌ای می‌دهند؟ نشان میدهد هر کلمه بیشتر در یک جمله با چه جفت هایی اشکار شده ؟

- چه چیزی هنوز کم است؟ (ترتیب / بافت جمله) ترتیب ! هنوز ترتیب از دید اسکیپ گرام مخفی میماند و برای آن باید PE را به اطلاعات خود راجع به هر توکن اضافه کنیم

</div>


<div dir="rtl">

## بخش ۱ـ د. Encoder کوچک

باید انجام دهید:
- embedding توکن + embedding موقعیت + یک یا دو لایه `TransformerEncoder`
- میانگین روی طول دنباله → لایهٔ کلاس
- **ماسک علّی نگذارید**
- شکل تنسور یک سند نمونه را چاپ کنید
- دقت آزمون را کنار مدل‌های قبلی بگذارید

</div>


In [ ]:
class TinyNewsEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, num_layers=2, num_classes=2, max_len=64):
        super().__init__()
        ### START CODE HERE ###
        pass
        ### END CODE HERE ###

    def forward(self, token_ids):
        ### START CODE HERE ###
        return None
        ### END CODE HERE ###


### START CODE HERE ###
# instantiate, pass one batch, print shapes:
# token ids, after embedding, after encoder, after classifier
# then train a little and report test accuracy
enc_test_acc = None
### END CODE HERE ###

print("encoder test acc:", enc_test_acc)


<div dir="rtl">

### توضیح بخش ۱ـ د

- شکل تنسورها:
- چرا اینجا causal mask نگذاشتید؟

</div>


<div dir="rtl">

## بخش ۱ـ ه. مقایسه و جمع‌بندی

جدول را کامل کنید و به سه سؤال PDF جواب دهید.

</div>


<div dir="rtl">

### جدول مقایسه پروژه ۱

| مدل | دقت آزمون | این مدل چه اطلاعاتی می‌بیند؟ |
|---|---|---|
| BoW |  |  |
| میانگین embedding |  |  |
| Encoder |  |  |

### سؤال‌ها

1. چرا بدون اطلاعات موقعیت، توجه ترتیب «اول / بعدی» را نمی‌فهمد؟
2. LayerNorm روی چه چیزی اعمال می‌شود؟
3. اگر به‌جای Encoder از RNN یک‌لایه استفاده می‌کردید، گلوگاه اصلی چه بود؟

**پاسخ:**

1.
2.
3.

</div>


<div dir="rtl">

# پروژه ۲. قلم نمایشنامه‌نویس: مدل زبانی کوچک

**هدف:** از متن خام، توکن بعدی را یاد بگیرید و بعد از یک پیشوند چند توکن تولید کنید.

**داده:** `Tiny Shakespeare` — فقط حدود ۱۲۰۰۰ تا ۲۰۰۰۰ نویسهٔ اول، نه کل فایل.

</div>


<div dir="rtl">

## بخش ۲ـ الف. متن، نرمال‌سازی و دو نگاه توکنایزری

باید انجام دهید:
- برش کوچک فایل را بخوانید.
- نرمال‌سازی سبک؛ تصمیم‌تان را بعداً توضیح دهید.
- همان دو خط را با `bert-base-uncased` و `gpt2` توکنایز کنید.
- برای مدل خودتان واژه‌نامه بسازید. توصیه: سطح **نویسه**.

</div>


In [ ]:
SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)
MAX_CHARS = 16000


### START CODE HERE ###
# download or read the file, keep only MAX_CHARS
# light normalization if you want
raw_text = None
text = None
### END CODE HERE ###

print("chars used:", None)
print(text[:400] if text else None)


In [ ]:
sample_lines = None  # two lines from the text

### START CODE HERE ###
# tokenize sample_lines with bert-base-uncased and gpt2
# print tokens, ids, and number of ids for each tokenizer
### END CODE HERE ###


In [ ]:
### START CODE HERE ###
# build a character-level vocab from `text`
# stoi / itos / stream of ids
stoi = None
itos = None
stream_ids = None
### END CODE HERE ###

print("char vocab size:", None)
print("stream length:", None)


<div dir="rtl">

### توضیح بخش ۲ـ الف

- چه نرمال‌سازی‌ای کردید و چرا؟
- BERT و GPT-2 روی همان دو خط چه فرقی داشتند؟
- چرا برای آموزش مدل خودتان واژه‌نامهٔ کوچک ساختید؟

</div>


<div dir="rtl">

## بخش ۲ـ ب. ساخت مثال next-token

باید انجام دهید:
- پنجره‌های طول ثابت، مثلاً ۳۲ یا ۶۴
- ورودی: همه به‌جز آخری ؛ هدف: همه به‌جز اولی
- یک مثال را از ID به متن برگردانید

</div>


In [ ]:
SEQ_LEN = 64

### START CODE HERE ###
x_lm = None  # [num_windows, SEQ_LEN]
y_lm = None  # [num_windows, SEQ_LEN]
### END CODE HERE ###

print("x_lm:", None)
print("y_lm:", None)
print("example input :", None)
print("example target:", None)


<div dir="rtl">

### توضیح بخش ۲ـ ب

- این کار چرا self-supervised است؟
- در مثال شما، مدل باید بعد از توکن اولِ ورودی چه چیزی را پیش‌بینی کند؟

</div>


<div dir="rtl">

## بخش ۲ـ ج. مدل decoder-style و ماسک علّی

باید انجام دهید:
- embedding توکن + embedding موقعیت + Transformer + لایه به اندازهٔ واژه‌نامه
- **ماسک علّی**
- چاپ ماسک طول ۸
- عبور یک batch و چاپ شکل logits: `[batch, length, vocab]`

</div>


In [ ]:
class TinyCharLM(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, num_layers=2, max_len=128):
        super().__init__()
        ### START CODE HERE ###
        pass
        ### END CODE HERE ###

    def forward(self, token_ids):
        ### START CODE HERE ###
        # build causal mask; True means blocked
        return None
        ### END CODE HERE ###


### START CODE HERE ###
# print an 8x8 causal mask
# pass a small batch and print logits shape
### END CODE HERE ###


<div dir="rtl">

### توضیح بخش ۲ـج

- `True` در ماسک یعنی چه؟
- Encoder را انتخاب کردید یا Decoder؟ چرا این انتخاب decoder-style است؟
- Decoder نسبت به Encoder چه زیرماژول اضافه‌ای دارد؟

</div>


<div dir="rtl">

## بخش ۲ـ د. آموزش کوتاه و تولید متن

باید انجام دهید:
- چند ده step با CrossEntropyLoss
- تابع تولید: پیشوند → توکن بعدی → اضافه کردن → تکرار
- شروع از `ROMEO:` یا `First Citizen:`
- حدود ۸۰ تا ۱۲۰ نویسه چاپ کنید

</div>


In [ ]:
### START CODE HERE ###
# train a little; print loss every few steps
### END CODE HERE ###


In [ ]:
def generate(model, prefix: str, max_new_tokens: int = 120) -> str:
    ### START CODE HERE ###
    return prefix
    ### END CODE HERE ###


print(generate(None, "ROMEO:"))


<div dir="rtl">

### توضیح بخش ۲ـ د

- loss چقدر کم شد؟
- متن تولیدی چه الگویی را تا حدی یاد گرفته و چه چیزی را بلد نیست؟

</div>


<div dir="rtl">

## بخش ۲ـ ه. یک آزمایش کوچک و جمع‌بندی

فقط **یکی** از سه آزمایش PDF را انجام دهید. بعد به دو سؤال آخر جواب دهید.

</div>


In [ ]:
### START CODE HERE ###
# optional: train briefly without causal mask
# or without positional embedding
# then write what you observed in the markdown cell below
### END CODE HERE ###


<div dir="rtl">

### توضیح بخش ۲ـ ه

کدام آزمایش را انتخاب کردید و چه دیدید؟

**سؤال‌ها**

1. چرا پروژهٔ ۱ به ماسک علّی نیاز نداشت و پروژهٔ ۲ نیاز داشت؟
2. فرق pretraining خودنظارتی این پروژه با fine-tune روی برچسب موضوع خبر چیست؟

**پاسخ:**

1.
2.

</div>


<div dir="rtl">

# چک‌لیست قبل از ارسال

- [ ] پروژهٔ ۱ روی حداکثر ۲۵۰+۱۰۰ سند از دو کلاس خواسته‌شده اجرا شده است.
- [ ] BoW، میانگین embedding و Encoder گزارش شده‌اند.
- [ ] دو سند اشتباه BoW و یک جفت جملهٔ دلخواه آمده است.
- [ ] چند جفت skip-gram ساخته شده است.
- [ ] پروژهٔ ۲ روی برش کوچک Tiny Shakespeare است.
- [ ] BERT و GPT-2 فقط برای مشاهدهٔ دو خط استفاده شده‌اند.
- [ ] یک مثال ورودی/هدف شیفت‌شده چاپ شده است.
- [ ] ماسک علّی چاپ شده و یک متن کوتاه تولید شده است.
- [ ] همهٔ سلول‌های «توضیح این بخش» پر شده‌اند.

</div>
